# Sesión 2 — Soluciones

> **Instructor:** este notebook contiene las soluciones completas de los ejercicios de sesión 2.  
> Compartir con los estudiantes después de la sesión, no durante.

---

In [ ]:
!pip install google-genai chromadb numpy Pillow python-dotenv --quiet

from google import genai
from google.genai import types
from pydantic import BaseModel
from typing import Optional, List
from PIL import Image, ImageDraw
import chromadb
import numpy as np
import json
import io
import os
import re

def load_api_key() -> str:
    try:
        from dotenv import load_dotenv
        load_dotenv()
        key = os.environ.get("GOOGLE_API_KEY")
        if key:
            print("API key cargada desde .env")
            return key
    except ImportError:
        pass
    try:
        from google.colab import userdata
        key = userdata.get("GOOGLE_API_KEY")
        if key:
            print("API key cargada desde Colab Secrets")
            return key
    except Exception:
        pass
    raise EnvironmentError(
        "No se encontró GOOGLE_API_KEY.\n"
        "Local: archivo .env con GOOGLE_API_KEY=AIza...\n"
        "Colab: panel de Secrets (ícono 🔑) con Notebook access activado."
    )

MODEL = "gemini-3.1-flash-lite-preview"
EMBED_MODEL = "gemini-embedding-2"

GOOGLE_API_KEY = load_api_key()
client = genai.Client(api_key=GOOGLE_API_KEY)

# ChromaDB
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection("circulares_sbs")
image_collection = chroma_client.get_or_create_collection("vouchers_financieros")

print(f"Setup completo. Circulares: {collection.count()} | Imágenes: {image_collection.count()}")

In [ ]:
# Funciones reutilizadas del notebook principal

def embed_texts(texts: list[str]) -> list[list[float]]:
    """Un embedding por llamada — gemini-embedding-2 no acepta batch."""
    return [
        client.models.embed_content(model=EMBED_MODEL, contents=text).embeddings[0].values
        for text in texts
    ]


def parse_md_to_articles(md_text: str, source_id: str, date: int) -> list[dict]:
    """date como int YYYYMM (ej: 202403). ChromaDB requiere int/float para filtros numéricos."""
    chunks = []
    pattern = re.compile(r'^#{1,3}\s*Art[ií]culo\s+[\w]+', re.MULTILINE | re.IGNORECASE)
    matches = list(pattern.finditer(md_text))
    if not matches:
        return [{"text": md_text.strip(), "source": source_id, "article": "completo", "date": date}]
    for i, match in enumerate(matches):
        start = match.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(md_text)
        chunk_text = md_text[start:end].strip()
        article_num = match.group(0).strip().split()[-1]
        chunks.append({"text": chunk_text, "source": source_id, "article": article_num, "date": date})
    return chunks


def retrieve(query: str, n_results: int = 3, where: dict = None) -> list[dict]:
    query_emb = embed_texts([query])[0]
    kwargs = {
        "query_embeddings": [query_emb],
        "n_results": min(n_results, max(collection.count(), 1)),
        "include": ["documents", "metadatas", "distances"]
    }
    if where:
        kwargs["where"] = where
    results = collection.query(**kwargs)
    return [
        {"text": doc, "metadata": meta, "distance": dist}
        for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0])
    ]


def make_voucher_pil(title, lines, color=(0, 102, 204)):
    img = Image.new("RGB", (480, 400), color="white")
    draw = ImageDraw.Draw(img)
    draw.rectangle([0, 0, 480, 60], fill=color)
    draw.text((240, 30), title, fill="white", anchor="mm")
    y = 80
    for key, value in lines:
        draw.text((20, y), f"{key}:", fill=(100, 100, 100))
        draw.text((460, y), value, fill=(20, 20, 20), anchor="ra")
        draw.line([20, y + 18, 460, y + 18], fill=(220, 220, 220), width=1)
        y += 32
    return img


class RAGResponse(BaseModel):
    answer: str
    sources: list[str]
    confidence_note: str


print("Funciones auxiliares definidas.")


---
## Solución — Ejercicio 1
### 5 queries y evaluación de retrieval

In [ ]:
# Asegurar que la colección tiene datos
if collection.count() == 0:
    print("Indexando datos primero...")
    BASE_DATA = None
    for candidate in ["data/circulares_sbs", "../data/circulares_sbs"]:
        if os.path.isdir(candidate):
            BASE_DATA = candidate
            break
    if BASE_DATA:
        # dates como int YYYYMM
        configs = [
            ("circular_B_2244_2024.md", "circular_SBS_B_2244_2024", 202403),
            ("circular_B_2251_2024.md", "circular_SBS_B_2251_2024", 202406),
            ("circular_B_2198_2022.md", "circular_SBS_B_2198_2022", 202201),
        ]
        for fname, sid, date in configs:
            path_md = os.path.join(BASE_DATA, fname)
            if os.path.exists(path_md):
                with open(path_md) as f:
                    text = f.read()
                chunks = parse_md_to_articles(text, sid, date)
                embeddings = embed_texts([c["text"] for c in chunks])
                collection.upsert(
                    ids=[f"{c['source']}__art_{c['article']}" for c in chunks],
                    embeddings=embeddings,
                    documents=[c["text"] for c in chunks],
                    metadatas=[{"source": c["source"], "article": c["article"], "date": c["date"]} for c in chunks]
                )
    print(f"Indexado. Total: {collection.count()} docs")

# Evaluación de queries
eval_queries = [
    ("¿Cuál es el umbral en USD para reportar transferencias internacionales?",
     "Art. 4 — circular_SBS_B_2244_2024"),
    ("¿Qué es una operación sospechosa según la SBS?",
     "Art. 2 — circular_SBS_B_2244_2024"),
    ("¿Cuánto tiempo tiene el Oficial de Cumplimiento para reportar a la UIF?",
     "Art. 7 — circular_SBS_B_2244_2024"),
    ("¿Cuáles son los límites de saldo en cuentas básicas de billeteras digitales?",
     "Art. 3 — circular_SBS_B_2251_2024"),
    ("¿Qué categoría crediticia corresponde a un deudor con 45 días de atraso?",
     "Art. 2 — circular_SBS_B_2198_2022"),
]

print(f"{'Query':<55} {'Top-1 recuperado':<45} {'Esperado':<45} {'OK?'}")
print("-" * 160)

for query, expected in eval_queries:
    results = retrieve(query, n_results=1)
    if results:
        top1 = f"Art. {results[0]['metadata']['article']} — {results[0]['metadata']['source']}"
        ok = "✓" if results[0]['metadata']['article'] in expected else "✗"
    else:
        top1 = "[sin resultados]"
        ok = "✗"
    print(f"{query[:54]:<55} {top1[:44]:<45} {expected[:44]:<45} {ok}")


---
## Solución — Ejercicio 2
### Retrieval mixto texto + imagen

In [ ]:
def describe_image(image: Image.Image) -> str:
    """Describe una imagen financiera con Gemini."""
    response = client.models.generate_content(
        model=MODEL,
        contents=[
            image,
            "Describe este documento financiero de forma detallada: tipo, entidad, montos, monedas, partes, concepto, estado. Texto plano, sin markdown."
        ],
        config=types.GenerateContentConfig(temperature=0.0)
    )
    return response.text


def index_image(image: Image.Image, image_id: str, metadata: dict) -> str:
    """Indexa una imagen describiendo primero con Gemini."""
    description = describe_image(image)
    embedding = embed_texts([description])[0]
    image_collection.upsert(
        ids=[image_id],
        embeddings=[embedding],
        documents=[description],
        metadatas=[{**metadata, "description": description[:500]}]
    )
    return description


# Crear e indexar el voucher de depósito en efectivo Interbank
voucher_interbank = make_voucher_pil(
    "INTERBANK - DEPOSITO EFECTIVO",
    [("Fecha", "03/05/2024"), ("Monto", "S/ 12,800.00"),
     ("Titular", "Rodriguez V. Carmen"), ("Agencia", "La Victoria"),
     ("Concepto", "Venta inmueble"), ("Estado", "COMPLETADO")],
    color=(0, 156, 68)
)

# Indexar si no existe
existing_ids = image_collection.get(ids=["voucher_interbank_deposito_004"]).get("ids", [])
if not existing_ids:
    print("Indexando voucher Interbank...")
    desc = index_image(
        voucher_interbank,
        "voucher_interbank_deposito_004",
        {"type": "deposito_efectivo", "date": 20240503, "amount_pen": "12800", "currency": "PEN"}
    )
    print(f"Descripción generada: {desc[:150]}...")
else:
    print("Voucher ya indexado.")

print(f"\nTotal imágenes indexadas: {image_collection.count()}")

In [ ]:
# Query mixta: texto normativo + contexto de imagen
query = "¿Un depósito en efectivo de S/ 12,800 requiere reporte? ¿Cuál es el umbral exacto?"

# Retrieval de texto normativo
text_results = retrieve(query, n_results=3)
print("=== Fragmentos normativos recuperados ===")
for r in text_results:
    print(f"  [{r['metadata']['source']} | Art. {r['metadata']['article']}]")
    print(f"  {r['text'][:100]}...")

# Retrieval de imágenes similares
q_emb = embed_texts([query])[0]
img_results = image_collection.query(
    query_embeddings=[q_emb],
    n_results=min(2, image_collection.count()),
    include=["documents", "metadatas", "distances"]
)
print("\n=== Imágenes similares recuperadas ===")
if img_results["documents"][0]:
    for doc, meta in zip(img_results["documents"][0], img_results["metadatas"][0]):
        print(f"  [{meta.get('type')} | {meta.get('date')}]")
        print(f"  {doc[:100]}...")

# Construir contexto combinado
normativa = "\n\n---\n\n".join([
    f"[{r['metadata']['source']} | Art. {r['metadata']['article']}]\n{r['text']}"
    for r in text_results
])
image_ctx = ""
if img_results["documents"][0]:
    image_ctx = "\n\nCONTEXTO DE OPERACIONES SIMILARES (imágenes):\n"
    for doc, meta in zip(img_results["documents"][0], img_results["metadatas"][0]):
        image_ctx += f"- [{meta.get('type')} | S/ {meta.get('amount_pen', '?')}]: {doc[:150]}\n"

# Llamada final a Gemini
augmented = f"""
Eres analista de compliance del sistema financiero peruano.
Responde ÚNICAMENTE basándote en la normativa proporcionada. Cita el artículo exacto.

=== NORMATIVA SBS ===
{normativa}
{image_ctx}

=== PREGUNTA ===
{query}
"""

response = client.models.generate_content(
    model=MODEL,
    contents=[voucher_interbank, augmented],
    config=types.GenerateContentConfig(
        temperature=0.0,
        max_output_tokens=400,
        response_mime_type="application/json",
        response_schema=RAGResponse
    )
)

result = json.loads(response.text)
print("\n=== RESPUESTA RAG MIXTO ===")
print(f"Respuesta: {result['answer']}")
print(f"Fuentes: {result['sources']}")
print(f"Nota: {result['confidence_note']}")

---
## Solución — Reto (4 opciones)
### Financial Profile Interrogator

In [ ]:
# Base del pipeline RAG para las 4 opciones

def rag_query_base(question: str, image=None, n_text_chunks=3, date_filter: int = None):
    """date_filter: int YYYYMM, ej: 202401 para solo normas desde enero 2024."""
    where = {"date": {"$gte": date_filter}} if date_filter else None
    chunks = retrieve(question, n_results=n_text_chunks, where=where)
    context = "\n\n---\n\n".join([
        f"[{c['metadata']['source']} | Art. {c['metadata']['article']}]\n{c['text']}"
        for c in chunks
    ])
    sources = [f"{c['metadata']['source']} · Art. {c['metadata']['article']}" for c in chunks]
    prompt = f"""
Eres analista de compliance SBS. Responde ÚNICAMENTE con la normativa proporcionada.
Cita la circular y el artículo exactos.

=== NORMATIVA ===
{context}

=== PREGUNTA ===
{question}
"""
    contents = [prompt] if image is None else [image, prompt]
    response = client.models.generate_content(
        model=MODEL, contents=contents,
        config=types.GenerateContentConfig(
            temperature=0.0, max_output_tokens=500,
            response_mime_type="application/json",
            response_schema=RAGResponse
        )
    )
    parsed = json.loads(response.text)
    if not parsed.get("sources"):
        parsed["sources"] = sources
    return parsed, chunks


In [ ]:
# === OPCIÓN A — Logging de queries a CSV ===
import csv
from datetime import datetime

class InterrogatorConLog:
    def __init__(self, log_file="query_log.csv"):
        self.log_file = log_file
        if not os.path.exists(log_file):
            with open(log_file, "w", newline="", encoding="utf-8") as f:
                csv.writer(f).writerow(["timestamp", "question", "top1_source", "answer", "confidence_note"])

    def analyze(self, client_profile: str, question: str, image=None, date_filter: int = None) -> dict:
        full_q = f"PERFIL:\n{client_profile}\n\nPREGUNTA:\n{question}"
        result, chunks = rag_query_base(full_q, image=image, date_filter=date_filter)
        top1 = chunks[0]["metadata"]["source"] + " Art. " + chunks[0]["metadata"]["article"] if chunks else "N/A"
        with open(self.log_file, "a", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow([
                datetime.now().isoformat(),
                question[:100],
                top1,
                result["answer"][:100],
                result["confidence_note"]
            ])
        return result


interrogator_a = InterrogatorConLog()
client_profile = """
Cliente: persona natural, exportador de textiles, 8 meses de antigüedad.
Movimiento: 3 transferencias internacionales en 48h, USD 45,000 total.
Destinos: Panamá e Islas Caimán.
"""
result = interrogator_a.analyze(
    client_profile,
    "¿Requiere reporte a la UIF según la normativa 2024?",
    date_filter=202401      # solo normas desde enero 2024
)
print(f"Respuesta: {result['answer'][:200]}")
print(f"Log guardado en: query_log.csv")


In [ ]:
# === OPCIÓN C — Reranking simple por longitud ===
# Hipótesis: chunks más largos tienen más contexto → mejor para responder

def rag_query_reranked(question: str, image=None, n_text_chunks=5):
    """
    Recupera n_text_chunks chunks y los reordena por longitud descendente
    antes de pasar los top-3 al LLM.
    """
    # Recuperar más chunks de los que vamos a usar
    chunks = retrieve(question, n_results=n_text_chunks)

    print(f"Antes del reranking (por similitud semántica):")
    for i, c in enumerate(chunks):
        print(f"  {i+1}. Art.{c['metadata']['article']} | {len(c['text'])} chars | dist={c['distance']:.4f}")

    # Reranking por longitud
    reranked = sorted(chunks, key=lambda c: len(c["text"]), reverse=True)
    top_3 = reranked[:3]

    print(f"\nDespués del reranking (por longitud):")
    for i, c in enumerate(top_3):
        print(f"  {i+1}. Art.{c['metadata']['article']} | {len(c['text'])} chars")

    # Llamada al LLM con los chunks rerankeados
    context = "\n\n---\n\n".join([
        f"[{c['metadata']['source']} | Art. {c['metadata']['article']}]\n{c['text']}"
        for c in top_3
    ])
    prompt = f"""
Eres analista de compliance SBS. Cita el artículo exacto.

=== NORMATIVA ===
{context}

=== PREGUNTA ===
{question}
"""
    response = client.models.generate_content(
        model=MODEL, contents=[prompt],
        config=types.GenerateContentConfig(
            temperature=0.0, max_output_tokens=400,
            response_mime_type="application/json",
            response_schema=RAGResponse
        )
    )
    return json.loads(response.text)


q = "¿Qué transferencias internacionales disparan una alerta automática y quién debe atenderla?"
print("=== RAG con reranking por longitud ===")
result_c = rag_query_reranked(q)
print(f"\nRespuesta: {result_c['answer'][:250]}")

In [ ]:
# === OPCIÓN D — Comparar respuesta con y sin imagen ===

voucher_bbva = make_voucher_pil(
    "BBVA - SWIFT INTERNACIONAL",
    [("Fecha", "28/04/2024"), ("Monto", "USD 15,000.00"),
     ("Banco destino", "Santander Uruguay"), ("Beneficiario", "C. Mendoza EIRL"),
     ("Proposito", "Pago proveedor"), ("Estado", "EN PROCESO")],
    color=(0, 40, 130)
)

question = "Analiza esta operación. ¿Requiere reporte a la UIF? ¿Bajo qué artículo?"

# Sin imagen: solo el texto de la pregunta
result_sin, _ = rag_query_base(question)
# Con imagen: Gemini puede ver el voucher
result_con, _ = rag_query_base(question, image=voucher_bbva)

print("=== COMPARACIÓN: SIN imagen vs CON imagen ===")
print()
print("SIN IMAGEN:")
print(f"  {result_sin['answer'][:250]}")
print(f"  Fuentes: {result_sin['sources']}")
print()
print("CON IMAGEN (voucher BBVA USD 15,000):")
print(f"  {result_con['answer'][:250]}")
print(f"  Fuentes: {result_con['sources']}")
print()
print("Observación esperada:")
print("  CON imagen: Gemini puede mencionar el monto exacto (USD 15,000), banco destino,")
print("  fecha, y propósito declarado del voucher — info que no estaba en el texto de la pregunta.")
print("  SIN imagen: respuesta más genérica, no puede referirse a datos específicos del voucher.")